# Query fan-out simulator and coverage checker

A free, Colab-ready notebook by **Lazarina Stoy, [MLforSEO](https://www.mlforseo.com/)**.

AI search systems don't answer your query in one go. Google describes AI Mode and AI Overviews as using a **[query fan-out](https://www.mlforseo.com/glossary-term/query-fan-out/)** technique:

* "It uses a 'query fan-out' technique, issuing multiple related searches concurrently across subtopics and multiple data sources and then brings those results together to provide an easy-to-understand response." ([Google, March 2025](https://blog.google/products/search/ai-mode-search/))
* "Under the hood, AI Mode uses our query fan-out technique, breaking down your question into subtopics and issuing a multitude of queries simultaneously on your behalf." ([Google, May 2025](https://blog.google/products/search/google-search-ai-mode-update/)). The same post says Deep Search takes this further and "can issue hundreds of searches".
* Google's Search Central documentation for site owners says both AI Overviews and [AI Mode](https://www.mlforseo.com/glossary-term/ai-mode/) "may use a 'query fan-out' technique" ([AI features and your website](https://developers.google.com/search/docs/appearance/ai-features)).

So the passages an AI answer draws on can be retrieved for any of the **sub-queries**, as well as for the query you typed. This notebook helps you prepare for that:

1. **Simulate the fan-out** for a seed query: either with an LLM (OpenAI, Anthropic or Gemini, your choice) using the prompt documented below, or with a free rule-based expansion that needs no key.
2. **Split your page(s) into passages** ([chunking](https://www.mlforseo.com/glossary-term/chunking/)), embed the passages and the sub-queries, and find the best-matching passage for each sub-query with [cosine similarity](https://www.mlforseo.com/glossary-term/cosine-similarity/).
3. **Rerank** the closest passages with a small cross-encoder that judges whether a passage actually answers the sub-query, then **label each sub-query** as covered, partial or gap, and export a coverage table and a gap list to write against.

### Please read this first: it's a simulation

Google does not publish the sub-queries it generates, how many there are, or how it chooses them. Nothing here shows you Google's real fan-out. The LLM produces *plausible* sub-queries (synthetic queries, see [synthetic queries](https://www.mlforseo.com/glossary-term/synthetic-queries/)), and the rule-based mode produces generic templates. Treat the output as a structured brainstorm of the questions a thorough answer would need to cover, and as a way to find thin spots in your content. A "covered" label means your page has a passage that is semantically close to the sub-query; it doesn't mean you will be retrieved or cited.

**What you need**

* A seed query and one or more page URLs (or the sample pages included here).
* Optional: an API key for OpenAI, Anthropic or Gemini for LLM fan-out. One call per seed query, typically a fraction of a cent with the small models set as defaults (check your provider's current pricing).
* Sample mode runs with no key on a CPU in about a minute.

In [1]:
# Install anything that's missing (Colab already has most of these)
import importlib, importlib.util, subprocess, sys
for pkg, mod in [("sentence-transformers", "sentence_transformers"), ("trafilatura", "trafilatura"),
                 ("beautifulsoup4", "bs4"), ("pandas", "pandas"), ("openpyxl", "openpyxl")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 180)

## 1. Settings

Leave `USE_SAMPLE = True` for the first run: it checks two sample pages from a fictional coffee site against the seed query *best espresso machine for beginners*, using the rule-based fan-out plus an illustrative example of LLM output.

For your own run: set `USE_SAMPLE = False`, write your seed query, list the URLs to check (one page, or a page plus its supporting pages), and pick `FANOUT_METHOD`:

* `"rules"`: free, offline, generic templates.
* `"llm"`: sub-queries from the provider in `LLM_PROVIDER`. Your key is read at runtime from Colab secrets (`OPENAI_API_KEY`, `ANTHROPIC_API_KEY` or `GEMINI_API_KEY`) or a hidden prompt, and never stored in the notebook.
* `"both"`: runs both and keeps the source of each sub-query.

In [2]:
USE_SAMPLE = True

SEED_QUERY = "best espresso machine for beginners"
PAGE_URLS = [
    # "https://www.example.com/your-page/",
]

FANOUT_METHOD = "both"          # "rules", "llm" or "both"
LLM_PROVIDER = "anthropic"      # "openai", "anthropic" or "gemini"
LLM_MODELS = {                  # small, cheap current models (October 2026); change freely
    "openai": "gpt-6-luna",
    "anthropic": "claude-haiku-5-5",
    "gemini": "gemini-3.5-flash-lite",
}
N_SUBQUERIES = 20

# Coverage scoring (section 4 explains both stages)
EMBEDDING_MODEL = "sentence-transformers/multi-qa-MiniLM-L6-cos-v1"   # stage 1: question -> passage similarity
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L6-v2"               # stage 2: does this passage answer it?
SCORING = "rerank"              # "rerank" (recommended) or "cosine" (embeddings only, faster)
TOP_K_PASSAGES = 5              # passages per sub-query sent to the reranker
COVERED_THRESHOLD = {"rerank": 6.0, "cosine": 0.75}
PARTIAL_THRESHOLD = {"rerank": 3.0, "cosine": 0.60}

REPO_RAW = "https://raw.githubusercontent.com/lazarinastoy/mlforseo-code/main/query-fan-out-coverage/"


def read_sample_text(name):
    """Read a sample file locally if present (cloned repo), otherwise from GitHub (Colab)."""
    local = Path("sample_data") / name
    if local.exists():
        return local.read_text(encoding="utf-8")
    r = requests.get(REPO_RAW + "sample_data/" + name, timeout=30)
    r.raise_for_status()
    return r.text

## 2. Simulate the fan-out

### Option A: rule-based expansion (no key)

The seed query is split into a **head term** (here *espresso machine*) and a **qualifier** (*for beginners*), and then dropped into templates for the facets a thorough answer usually covers. There are two template sets: a product set (definition, types, buying criteria, price, recommendations, comparisons, problems, how to use, maintenance, accessories, "is it worth it") used when the seed has cues like *best*, *review* or *price*, and a concept set (definition, how it works, examples, why it matters, best practices, tools, mistakes, measurement) for everything else. It's crude and English-only, and it knows nothing about your niche, so it can't name brands or products. It's still a useful baseline, and it costs nothing.

In [3]:
LEADING_MODIFIERS = (r"^((best|top rated|top|cheapest|cheap|affordable|good)\s+"
                     r"|(how to (choose|buy|pick)|what is|what are|what's|how does|how do|why is|why does|explain)\s+"
                     r"((a|an|the)\s+)?)")
TRAILING_QUALIFIER = r"\s+((for|under|with|without|in|near)\s.+|\d{4})$"
PRODUCT_CUES = r"\b(best|top|buy|cheap|cheapest|price|prices|review|reviews|deal|deals|under \d+)\b"

# Product-style seeds ("best espresso machine for beginners")
PRODUCT_TEMPLATES = [
    ("definition",      "what is a {head}"),
    ("how it works",    "how does a {head} work"),
    ("types",           "types of {head}"),
    ("buying criteria", "what to look for in a {head} {qualifier}"),
    ("buying criteria", "{head} features that matter"),
    ("price",           "how much does a good {head} cost"),
    ("recommendations", "best {head} {qualifier}"),
    ("recommendations", "top rated {head} brands"),
    ("comparison",      "{head} alternatives"),
    ("comparison",      "cheap vs expensive {head}"),
    ("reviews",         "{head} reviews"),
    ("problems",        "common {head} problems"),
    ("problems",        "{head} mistakes to avoid {qualifier}"),
    ("how to use",      "how to use a {head}"),
    ("maintenance",     "how to clean a {head}"),
    ("maintenance",     "how long does a {head} last"),
    ("accessories",     "{head} accessories you need"),
    ("worth it",        "is a {head} worth it"),
]

# Concept-style seeds ("what is query fan-out in ai mode")
CONCEPT_TEMPLATES = [
    ("definition",      "what is {head}"),
    ("definition",      "{head} explained simply"),
    ("how it works",    "how does {head} work"),
    ("examples",        "{head} examples"),
    ("why it matters",  "why does {head} matter"),
    ("context",         "{head} {qualifier}"),
    ("how to",          "how to get started with {head}"),
    ("best practices",  "{head} best practices"),
    ("comparison",      "{head} vs alternatives"),
    ("tools",           "{head} tools"),
    ("problems",        "common {head} mistakes"),
    ("measurement",     "how to measure {head}"),
    ("audience",        "{head} for beginners"),
    ("history",         "history of {head}"),
    ("freshness",       "{head} latest updates"),
]


def split_seed(seed):
    """Split a seed into (head term, qualifier): 'best espresso machine for beginners' ->
    ('espresso machine', 'for beginners')."""
    seed = seed.lower().strip()
    qualifier_match = re.search(TRAILING_QUALIFIER, seed)
    qualifier = qualifier_match.group(0).strip() if qualifier_match else ""
    head = re.sub(TRAILING_QUALIFIER, "", seed)
    for _ in range(2):                                   # e.g. "what is the best ..."
        head = re.sub(LEADING_MODIFIERS, "", head).strip()
    head = re.sub(r"\s+(work|works|mean|means)$", "", head)
    return head, qualifier


def rule_based_fanout(seed, limit=N_SUBQUERIES):
    head, qualifier = split_seed(seed)
    templates = PRODUCT_TEMPLATES if re.search(PRODUCT_CUES, seed.lower()) else CONCEPT_TEMPLATES
    rows, seen = [], {seed.lower().strip()}
    for facet, template in templates:
        q = re.sub(r"\s+", " ", template.format(head=head, qualifier=qualifier)).strip()
        q = re.sub(r"\ba (?=[aeiou])", "an ", q)        # "a espresso" -> "an espresso"
        if q not in seen and q != head:
            seen.add(q)
            rows.append({"sub_query": q, "facet": facet, "source": "rules"})
    return rows[:limit]


print("head term:", split_seed(SEED_QUERY))
pd.DataFrame(rule_based_fanout(SEED_QUERY))

head term: ('espresso machine', 'for beginners')


,sub_query,facet,source
0,what is an espresso machine,definition,rules
1,how does an espresso machine work,how it works,rules
2,types of espresso machine,types,rules
3,what to look for in an espresso machine for beginners,buying criteria,rules
4,espresso machine features that matter,buying criteria,rules
5,how much does a good espresso machine cost,price,rules
6,top rated espresso machine brands,recommendations,rules
7,espresso machine alternatives,comparison,rules
8,cheap vs expensive espresso machine,comparison,rules
9,espresso machine reviews,reviews,rules


### Option B: LLM fan-out (pluggable provider)

The prompt below asks the model to behave like the query planner of an AI search system: break the seed query into the sub-queries it would run to write a complete, well-sourced answer. The facets it asks for (reformulations, implicit needs, comparisons, entity expansions, follow-ups, specifics) mirror Google's own description ("subtopics", "multiple data sources") and the way practitioners have reasoned about fan-out. They're our assumptions, not a documented Google spec.

**The prompt** (edit `FANOUT_PROMPT` to fit your niche, market or language):

In [4]:
FANOUT_PROMPT = """You are the query planner inside an AI search engine (similar to Google's AI Mode).
A user has searched for: "{seed}"

Before writing an answer, the engine "fans out" the query: it runs several related searches
across the subtopics a complete, trustworthy answer would need. List the {n} searches you would run.

Cover a mix of these facets:
- reformulation: the same need in different words
- implicit need: things the user will need to know but didn't ask (prerequisites, hidden questions)
- comparison: options, alternatives, X vs Y
- entity expansion: specific products, brands, standards, people or places relevant to the query
- specifics: numbers, specs, prices, timeframes, criteria
- follow-up: what the user is likely to ask next

Rules: write each sub-query the way a person would type it into a search box (short, lowercase,
no quotation marks). Don't repeat the seed query. Be specific to the topic.

Return ONLY JSON in this shape, with no other text:
{{"sub_queries": [{{"sub_query": "...", "facet": "...", "intent": "informational|commercial|transactional|navigational"}}]}}
"""


def get_secret(name):
    """Colab secrets first, then a hidden prompt. Never hard-code keys."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        from getpass import getpass
        return getpass(f"{name}: ")


def call_llm(prompt, provider=LLM_PROVIDER, model=None):
    """Send one prompt to OpenAI, Anthropic or Gemini and return the text reply."""
    model = model or LLM_MODELS[provider]
    if provider == "openai":
        from openai import OpenAI                      # pip install openai
        client = OpenAI(api_key=get_secret("OPENAI_API_KEY"))
        return client.responses.create(model=model, input=prompt).output_text
    if provider == "anthropic":
        import anthropic                               # pip install anthropic
        client = anthropic.Anthropic(api_key=get_secret("ANTHROPIC_API_KEY"))
        msg = client.messages.create(model=model, max_tokens=4000,
                                     messages=[{"role": "user", "content": prompt}])
        return "".join(block.text for block in msg.content if block.type == "text")
    if provider == "gemini":
        from google import genai                       # pip install google-genai
        client = genai.Client(api_key=get_secret("GEMINI_API_KEY"))
        resp = client.models.generate_content(model=model, contents=prompt,
                                              config={"response_mime_type": "application/json"})
        return resp.text
    raise ValueError("LLM_PROVIDER must be 'openai', 'anthropic' or 'gemini'")


def parse_fanout_json(text):
    """Pull the JSON object out of the reply, even if the model wrapped it in a code fence."""
    match = re.search(r"\{.*\}", text, flags=re.S)
    data = json.loads(match.group(0) if match else text)
    return [{"sub_query": str(d["sub_query"]).strip().lower(), "facet": d.get("facet", ""),
             "source": "llm"} for d in data.get("sub_queries", []) if d.get("sub_query")]


def llm_fanout(seed, n=N_SUBQUERIES):
    package = {"openai": "openai", "anthropic": "anthropic", "gemini": "google-genai"}[LLM_PROVIDER]
    try:
        importlib.import_module("google.genai" if package == "google-genai" else package)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
    return parse_fanout_json(call_llm(FANOUT_PROMPT.format(seed=seed, n=n)))

### Build the sub-query list

In sample mode, the "LLM" rows come from `sample_data/example_llm_fanout.json`: an **illustrative example written by hand** to show the format and the kind of sub-queries models tend to return. It is not a real model response. With `USE_SAMPLE = False` and `FANOUT_METHOD = "llm"` or `"both"`, the notebook calls your chosen provider instead.

In [5]:
subqueries = []
if FANOUT_METHOD in ("rules", "both"):
    subqueries += rule_based_fanout(SEED_QUERY)

if FANOUT_METHOD in ("llm", "both"):
    if USE_SAMPLE:
        example = json.loads(read_sample_text("example_llm_fanout.json"))
        subqueries += [{"sub_query": d["sub_query"], "facet": d["facet"], "source": "llm (illustrative example)"}
                       for d in example["sub_queries"]]
    else:
        subqueries += llm_fanout(SEED_QUERY)

subqueries = pd.DataFrame(subqueries).drop_duplicates("sub_query", ignore_index=True)
print(f"{len(subqueries)} sub-queries for '{SEED_QUERY}'")
subqueries

35 sub-queries for 'best espresso machine for beginners'


,sub_query,facet,source
0,what is an espresso machine,definition,rules
1,how does an espresso machine work,how it works,rules
2,types of espresso machine,types,rules
3,what to look for in an espresso machine for beginners,buying criteria,rules
4,espresso machine features that matter,buying criteria,rules
5,how much does a good espresso machine cost,price,rules
6,top rated espresso machine brands,recommendations,rules
7,espresso machine alternatives,comparison,rules
8,cheap vs expensive espresso machine,comparison,rules
9,espresso machine reviews,reviews,rules


## 3. Load the page(s) and split them into passages

For live pages the notebook downloads each URL and extracts the main content with [trafilatura](https://trafilatura.readthedocs.io/) (menus, footers and cookie banners are dropped), falling back to BeautifulSoup if that fails. Pages that need JavaScript to render may come back empty; paste their text into a `.txt` file instead (first lines `URL: ...` and `TITLE: ...`, then the content) and add the file name to `LOCAL_TEXT_FILES`.

Passages are built the way retrieval systems tend to see content: headings start a new passage and are kept as context, short paragraphs in the same section are merged until a passage has at least `MIN_WORDS`, and long ones are split at sentence boundaries at about `MAX_WORDS`. See [passage ranking](https://www.mlforseo.com/glossary-term/passage-ranking/) for why passages matter.

In [6]:
MIN_WORDS, MAX_WORDS = 40, 160
LOCAL_TEXT_FILES = []          # optional: your own .txt files, same format as the sample pages
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; MLforSEO-fanout-coverage/1.0)"}


def fetch_page(url):
    """Download a page and return (title, main text with markdown-style headings)."""
    import trafilatura
    from bs4 import BeautifulSoup
    html = requests.get(url, headers=HEADERS, timeout=30).text
    text = trafilatura.extract(html, output_format="markdown", include_comments=False,
                               include_tables=True, include_formatting=True) or ""
    soup = BeautifulSoup(html, "html.parser")
    title = soup.title.get_text(strip=True) if soup.title else url
    if len(text.split()) < 50:      # fallback: headings, paragraphs and list items
        parts = []
        for el in soup.find_all(["h1", "h2", "h3", "h4", "p", "li"]):
            t = el.get_text(" ", strip=True)
            if t:
                parts.append(("#" * int(el.name[1]) + " " + t) if el.name.startswith("h") else t)
        text = "\n\n".join(parts)
    return title, text


def parse_text_file(raw):
    """Sample/local format: 'URL: ...' and 'TITLE: ...' lines, then the content."""
    url = re.search(r"^URL:\s*(.+)$", raw, flags=re.M).group(1).strip()
    title = re.search(r"^TITLE:\s*(.+)$", raw, flags=re.M).group(1).strip()
    body = re.sub(r"^(URL|TITLE):.*$", "", raw, flags=re.M)
    return url, title, body


def split_sentences(text):
    return re.split(r"(?<=[.!?])\s+", text)


def chunk_page(url, title, text):
    """Heading-aware passages of roughly MIN_WORDS..MAX_WORDS words."""
    passages, heading, buffer = [], title, []

    def flush():
        if buffer:
            passages.append({"url": url, "heading": heading, "text": " ".join(buffer)})
            buffer.clear()

    for block in re.split(r"\n\s*\n", text):
        block = block.strip()
        if not block:
            continue
        if block.startswith("#"):                      # a heading starts a new section
            flush()
            heading = block.lstrip("#").strip()
            continue
        block = re.sub(r"\s+", " ", block)
        if len(block.split()) > MAX_WORDS:             # long paragraph: split at sentences
            flush()
            for sentence in split_sentences(block):
                buffer.append(sentence)
                if len(" ".join(buffer).split()) >= MAX_WORDS:
                    flush()
            continue
        buffer.append(block)
        if len(" ".join(buffer).split()) >= MIN_WORDS:
            flush()
    flush()
    return passages


pages = []      # (url, title, text)
if USE_SAMPLE:
    for name in ["pages/espresso-machine-buying-guide.txt", "pages/how-to-descale-espresso-machine.txt"]:
        pages.append(parse_text_file(read_sample_text(name)))
else:
    for url in PAGE_URLS:
        try:
            title, text = fetch_page(url)
            pages.append((url, title, text))
        except Exception as e:
            print("Could not fetch", url, "->", e)
    for path in LOCAL_TEXT_FILES:
        pages.append(parse_text_file(Path(path).read_text(encoding="utf-8")))

passages = pd.DataFrame([p for url, title, text in pages for p in chunk_page(url, title, text)])
passages["words"] = passages["text"].str.split().str.len()
print(f"{len(pages)} page(s) -> {len(passages)} passages "
      f"(median {int(passages['words'].median())} words)")
passages[["url", "heading", "words"]]

2 page(s) -> 17 passages (median 52 words)


,url,heading,words
0,https://www.homebaristahq.example/guides/best-espresso-machine-for-beginners/,The Best Espresso Machines for Beginners,64
1,https://www.homebaristahq.example/guides/best-espresso-machine-for-beginners/,What makes an espresso machine good for beginners?,47
2,https://www.homebaristahq.example/guides/best-espresso-machine-for-beginners/,What makes an espresso machine good for beginners?,64
3,https://www.homebaristahq.example/guides/best-espresso-machine-for-beginners/,Types of espresso machines,49
4,https://www.homebaristahq.example/guides/best-espresso-machine-for-beginners/,Types of espresso machines,67
5,https://www.homebaristahq.example/guides/best-espresso-machine-for-beginners/,"Single boiler, heat exchanger or dual boiler?",74
6,https://www.homebaristahq.example/guides/best-espresso-machine-for-beginners/,How much should you spend?,61
7,https://www.homebaristahq.example/guides/best-espresso-machine-for-beginners/,Our recommendations,59
8,https://www.homebaristahq.example/guides/best-espresso-machine-for-beginners/,Our recommendations,44
9,https://www.homebaristahq.example/guides/best-espresso-machine-for-beginners/,Steaming milk on a beginner machine,99


## 4. Score coverage

Scoring happens in two stages, the way many retrieval systems work:

1. **Retrieve.** Every sub-query and every passage is turned into an [embedding](https://www.mlforseo.com/glossary-term/embedding/) with `multi-qa-MiniLM-L6-cos-v1`, a small free model trained to match questions with passages. Each passage is embedded with its heading, because a heading tells a retrieval system what the passage is about. For each sub-query the `TOP_K_PASSAGES` most similar passages (by [cosine similarity](https://www.mlforseo.com/glossary-term/cosine-similarity/)) go through to stage 2.
2. **Rerank.** A small cross-encoder (`ms-marco-MiniLM-L6-v2`) reads each sub-query and candidate passage *together* and scores how likely the passage answers it. This matters because on a focused page every passage is "about" the topic, so cosine similarity alone scores almost everything as close. In testing on the sample, cosine rated *is buying a used espresso machine worth it* as covered (the page never mentions used machines); the reranker didn't.

Each sub-query keeps its single best passage across all your pages and gets a label:

| Label | Rerank score (default) | Cosine only (`SCORING = "cosine"`) | Meaning |
|---|---|---|---|
| **covered** | ≥ 6 | ≥ 0.75 | A passage answers this directly |
| **partial** | 3 to 6 | 0.60 to 0.75 | The topic is touched on but not answered head-on |
| **gap** | < 3 | < 0.60 | Nothing on the page(s) answers it |

The rerank score is the model's raw relevance score (roughly −11 to +11 here; higher is better). The thresholds were set by reading the sample results, so read 10 to 15 of your own rows and adjust them if the labels look off.

In [7]:
from sentence_transformers import CrossEncoder

model = SentenceTransformer(EMBEDDING_MODEL, device="cpu")
passage_inputs = (passages["heading"] + ". " + passages["text"]).tolist()
P = model.encode(passage_inputs, normalize_embeddings=True, batch_size=32, show_progress_bar=False)
Q = model.encode(subqueries["sub_query"].tolist(), normalize_embeddings=True, show_progress_bar=False)
cos = Q @ P.T                                  # sub-queries x passages

if SCORING == "rerank":
    reranker = CrossEncoder(RERANKER_MODEL, device="cpu")
    best, best_score = [], []
    for i, query in enumerate(subqueries["sub_query"]):
        candidates = np.argsort(-cos[i])[:TOP_K_PASSAGES]
        scores = reranker.predict([(query, passage_inputs[j]) for j in candidates], show_progress_bar=False)
        best.append(candidates[int(np.argmax(scores))])
        best_score.append(float(np.max(scores)))
    best, best_score = np.array(best), np.array(best_score)
else:
    best, best_score = cos.argmax(axis=1), cos.max(axis=1)


def status(score):
    if score >= COVERED_THRESHOLD[SCORING]:
        return "covered"
    return "partial" if score >= PARTIAL_THRESHOLD[SCORING] else "gap"


coverage = subqueries.copy()
coverage["score"] = np.round(best_score, 2)
coverage["status"] = coverage["score"].map(status)
coverage["cosine_similarity"] = np.round(cos[np.arange(len(best)), best], 3)
coverage["best_url"] = passages["url"].values[best]
coverage["best_heading"] = passages["heading"].values[best]
coverage["best_passage"] = [t[:220] + ("..." if len(t) > 220 else "") for t in passages["text"].values[best]]
coverage = coverage.sort_values("score", ascending=False, ignore_index=True)

order = ["covered", "partial", "gap"]
summary = coverage["status"].value_counts().reindex(order, fill_value=0)
overall = (summary["covered"] + 0.5 * summary["partial"]) / len(coverage)
print(f"Seed query: {SEED_QUERY}   (scoring: {SCORING})")
print(f"Coverage score: {overall:.0%}  ({summary['covered']} covered, {summary['partial']} partial, "
      f"{summary['gap']} gaps out of {len(coverage)} sub-queries)\n")
print(pd.crosstab(coverage["source"], coverage["status"]).reindex(columns=order, fill_value=0))

coverage[["sub_query", "facet", "source", "score", "status", "cosine_similarity", "best_heading"]]

Seed query: best espresso machine for beginners   (scoring: rerank)
Coverage score: 54%  (11 covered, 16 partial, 8 gaps out of 35 sub-queries)

status                      covered  partial  gap
source                                           
llm (illustrative example)        6        6    6
rules                             5       10    2


,sub_query,facet,source,score,status,cosine_similarity,best_heading
0,how much does a good beginner espresso machine cost,price,llm (illustrative example),9.23,covered,0.865,The Best Espresso Machines for Beginners
1,how much does a good espresso machine cost,price,rules,8.96,covered,0.818,The Best Espresso Machines for Beginners
2,types of espresso machine,types,rules,8.94,covered,0.748,Types of espresso machines
3,single boiler vs dual boiler espresso machine,comparison,llm (illustrative example),8.78,covered,0.801,"Single boiler, heat exchanger or dual boiler?"
4,what to look for in an espresso machine for beginners,buying criteria,rules,7.48,covered,0.769,What makes an espresso machine good for beginners?
5,is a 15 bar espresso machine good,specifications,llm (illustrative example),7.34,covered,0.684,What makes an espresso machine good for beginners?
6,what to look for in a beginner espresso machine,buying criteria,llm (illustrative example),7.15,covered,0.761,What makes an espresso machine good for beginners?
7,what is an espresso machine,definition,rules,7.07,covered,0.727,What makes an espresso machine good for beginners?
8,how to steam milk with an espresso machine steam wand,follow-up,llm (illustrative example),7.01,covered,0.665,Steaming milk on a beginner machine
9,how does an espresso machine work,how it works,rules,6.80,covered,0.658,What makes an espresso machine good for beginners?


In [8]:
# What the best match looks like for a few sub-queries: read these to sanity-check the thresholds
for _, row in pd.concat([coverage.head(2), coverage.iloc[[len(coverage) // 2]], coverage.tail(2)]).iterrows():
    print(f"[{row.status} {row.score:.2f}] {row.sub_query}")
    print(f"    best passage ({row.best_heading}): {row.best_passage}\n")

[covered 9.23] how much does a good beginner espresso machine cost
    best passage (The Best Espresso Machines for Beginners): Buying your first espresso machine is exciting, and a little overwhelming. There are hundreds of models, the prices range from under £100 to well over £2,000, and every product page promises café-quality coffee. This gui...

[covered 8.96] how much does a good espresso machine cost
    best passage (The Best Espresso Machines for Beginners): Buying your first espresso machine is exciting, and a little overwhelming. There are hundreds of models, the prices range from under £100 to well over £2,000, and every product page promises café-quality coffee. This gui...

[partial 4.71] espresso accessories every beginner needs
    best passage (What makes an espresso machine good for beginners?): For most beginners that means a semi-automatic machine with a single boiler or a thermoblock heater, a standard 58mm or 54mm portafilter, and a steam wand that is easy to cont

## 5. The gap list and per-page view

The gap list is your to-do list: gaps first (nothing covers them), then partials (something is there but doesn't answer the sub-query directly). For each one, decide whether it belongs on this page (a new section or a sharper paragraph), on a supporting page you link to, or nowhere (not every sub-query fits your audience).

When you check several pages, the per-page table shows which page "owns" which sub-queries, which helps with internal linking and avoiding two pages answering the same thing.

In [9]:
gaps = (coverage[coverage["status"] != "covered"]
        .assign(priority=lambda d: d["status"].map({"gap": 1, "partial": 2}))
        .sort_values(["priority", "score"])
        .drop(columns="priority")
        .reset_index(drop=True))
gaps["suggested_action"] = np.where(
    gaps["status"] == "gap",
    "Add a section or passage that answers this directly (or cover it on a linked page)",
    "Sharpen the closest passage: answer the sub-query in its first sentence, use its key terms",
)
print(f"{len(gaps)} sub-queries to work on")
display(gaps[["sub_query", "facet", "status", "score", "best_heading", "suggested_action"]])

per_page = (coverage.groupby(["best_url", "status"]).size().unstack(fill_value=0)
            .reindex(columns=order, fill_value=0))
per_page

24 sub-queries to work on


,sub_query,facet,status,score,best_heading,suggested_action
0,breville bambino vs gaggia classic,entity expansion,gap,-11.41,Our recommendations,Add a section or passage that answers this directly (or cover it on a linked page)
1,do I need a separate grinder for an espresso machine,implicit need,gap,1.14,How much should you spend?,Add a section or passage that answers this directly (or cover it on a linked page)
2,can you make an americano with an espresso machine,follow-up,gap,1.36,What makes an espresso machine good for beginners?,Add a section or passage that answers this directly (or cover it on a linked page)
3,how long does a home espresso machine last,follow-up,gap,1.96,What you need,Add a section or passage that answers this directly (or cover it on a linked page)
4,how to dial in espresso as a beginner,follow-up,gap,2.33,What makes an espresso machine good for beginners?,Add a section or passage that answers this directly (or cover it on a linked page)
5,what grind size for espresso,implicit need,gap,2.41,What makes an espresso machine good for beginners?,Add a section or passage that answers this directly (or cover it on a linked page)
6,top rated espresso machine brands,recommendations,gap,2.76,Types of espresso machines,Add a section or passage that answers this directly (or cover it on a linked page)
7,espresso machine reviews,reviews,gap,2.92,What makes an espresso machine good for beginners?,Add a section or passage that answers this directly (or cover it on a linked page)
8,espresso machine alternatives,comparison,partial,3.01,Types of espresso machines,"Sharpen the closest passage: answer the sub-query in its first sentence, use its key t..."
9,what is PID temperature control on an espresso machine,specifications,partial,3.10,How much should you spend?,"Sharpen the closest passage: answer the sub-query in its first sentence, use its key t..."


status,covered,partial,gap
best_url,,,
https://www.homebaristahq.example/guides/best-espresso-machine-for-beginners/,10,12,7
https://www.homebaristahq.example/guides/how-to-descale-an-espresso-machine/,1,4,1


## 6. Export

* `fanout_coverage.csv`: every sub-query with its facet, source, best score, status, best page, heading and passage.
* `fanout_gaps.csv`: gaps and partials with a suggested action.
* `fanout_coverage.xlsx`: both, plus the passages, as tabs. In Google Sheets use **File → Import → Upload**.

In [10]:
slug = re.sub(r"[^a-z0-9]+", "-", SEED_QUERY.lower()).strip("-")
coverage.insert(0, "seed_query", SEED_QUERY)
gaps.insert(0, "seed_query", SEED_QUERY)

coverage.to_csv(f"fanout_coverage_{slug}.csv", index=False)
gaps.to_csv(f"fanout_gaps_{slug}.csv", index=False)
with pd.ExcelWriter(f"fanout_coverage_{slug}.xlsx") as xl:
    coverage.to_excel(xl, sheet_name="Coverage", index=False)
    gaps.to_excel(xl, sheet_name="Gaps", index=False)
    passages.to_excel(xl, sheet_name="Passages", index=False)
print("Saved:", f"fanout_coverage_{slug}.csv,", f"fanout_gaps_{slug}.csv,", f"fanout_coverage_{slug}.xlsx")
# In Colab, download with:  from google.colab import files; files.download(f"fanout_coverage_{slug}.xlsx")

Saved: fanout_coverage_best-espresso-machine-for-beginners.csv, fanout_gaps_best-espresso-machine-for-beginners.csv, fanout_coverage_best-espresso-machine-for-beginners.xlsx


## How to read the results (and the limits)

* **On the sample**, the buying guide answers the core of the query directly (budgets, machine types, boilers, pressure, steaming milk) and the descaling page picks up the cleaning sub-query, which is why it's worth checking a page *together with* its supporting pages. The gaps are things a beginner needs that the guide never mentions: a grinder, grind size, dialling in, named models, how long a machine lasts, and drinks like an americano.
* **Partials are worth a closer look.** Some are near-misses that a sharper sentence would fix (the capsule-machine paragraph never compares capsules with espresso machines head-on). Some are answers split across two passages: *semi automatic vs super automatic* is covered, but in two separate paragraphs, so no single passage answers the comparison. That is exactly what a passage-level retrieval system sees too.
* **Rules vs LLM.** Rule-based sub-queries are generic, so they tend to be either easy to cover or irrelevant. LLM sub-queries are more specific (named products, real implicit needs) and usually find the more useful gaps.
* **Run it a few times.** LLM fan-out isn't deterministic, and real systems vary their sub-queries too. Sub-queries that keep coming back across runs and providers are the ones worth writing for.

**Limits, honestly:**

* This simulates fan-out. Google doesn't publish its sub-queries, and AI Mode also personalises and draws on sources beyond web pages.
* The scores measure relevance, not quality.  A passage can be "covered" and still be vague, outdated or wrong. Read the matches.
* Thresholds depend on the models and on how long your passages are. The defaults were set on the sample.
* Both models are English. For other languages try `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` with `SCORING = "cosine"` (or a multilingual cross-encoder) and re-check the thresholds.
* The sample pages are fictional, written for this notebook.

## What to do next

* Read [How entities power query fan-out in AI search](https://www.mlforseo.com/machine-learning-implementation-guides/ai-search-optimisation/how-entities-power-query-fan-out-in-ai-search/) and [What are synthetic queries?](https://www.mlforseo.com/ml-fundamentals/what-are-synthetic-queries-semantic-seo-ai-search/)
* Build the query research skills behind this in the [Semantic ML-enabled Keyword Research course](https://www.mlforseo.com/courses/semantic-ml-enabled-keyword-research/), and the content side in [AI Search Optimisation & Agentic SEO](https://www.mlforseo.com/courses/ai-search-optimisation-agentic-seo/).
* More free notebooks: the [MLforSEO resources library](https://www.mlforseo.com/resources-library/) and the [mlforseo-code repo](https://github.com/lazarinastoy/mlforseo-code).

*Made by Lazarina Stoy, MLforSEO. MIT licence.*